In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
import re
import matplotlib.pyplot as plt
import seaborn as sns
import string
import os

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

## 1. Đọc và Kiểm tra dữ liệu

In [ ]:
# 1. Đọc dữ liệu từ file csv
df_fake = pd.read_csv('../data/raw/Fake.csv')
df_true = pd.read_csv('../data/raw/True.csv')

# 2. Gán nhãn (1: Fake, 0: True)
df_fake['label'] = 1
df_true['label'] = 0

# 3. Gộp 2 DataFrame và xáo trộn
df = pd.concat([df_fake, df_true], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Đã đọc và gộp dữ liệu thành công.")
# Hợp nhất tiêu đề và nội dung để có dữ liệu văn bản đầy đủ (tùy chọn)
df['text'] = df['title'] + " " + df['text']

# df['text'] = df['text'].str.replace(r'^.*?\(reuters\) - ', '', regex=True, flags=re.IGNORECASE)
# df = df.dropna(subset=['text'])
# df['text'] = df['text'].astype(str)

print(df.head())

In [ ]:
# Thống kê quy mô và cấu trúc
num_samples, num_features = df.shape
label_counts = df['label'].value_counts()

print(f"=== THỐNG KÊ BAN ĐẦU ===")
print(f"Tổng số mẫu: {num_samples}")
print(f"Số đặc trưng: {num_features}")
print(f"Danh sách cột: {list(df.columns)}")
print("-" * 30)
print(f"Số lượng tin giả (Label 1): {label_counts[1]}")
print(f"Số lượng tin thật (Label 0): {label_counts[0]}")
print(f"Tỷ lệ tin giả: {(label_counts[1]/num_samples)*100:.2f}%")

In [ ]:
print("=== KIỂM TRA ĐỘ SẠCH ===")
# 1. Kiểm tra giá trị thiếu (NaN)
print("Giá trị thiếu trên từng cột:")
print(df.isnull().sum())

# 2. Kiểm tra dòng trùng lặp hoàn toàn
print(f"\nSố dòng trùng lặp hoàn toàn: {df.duplicated().sum()}")

# 3. Kiểm tra trùng lặp nội dung văn bản (Quan trọng nhất)
print(f"Số dòng trùng lặp nội dung 'text': {df.duplicated(subset=['text']).sum()}")

## 2. Làm sạch dữ liệu cơ bản

In [ ]:
from nltk.classify import scikitlearn
# 1. Loại bỏ các cột không cần thiết (Tránh Data Leakage)
cols_to_drop = ['subject', 'date']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True, errors='ignore')

# 2. Xử lý trùng lặp dựa trên nội dung bài báo
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

# 3. Loại bỏ giá trị thiếu (NaN) và các dòng text rỗng/chỉ chứa khoảng trắng
df = df.dropna().reset_index(drop=True)
df = df[df['text'].str.strip() != ""].reset_index(drop=True)

# 4. Xóa dấu hiệu nguồn tin (ví dụ: "WASHINGTON (Reuters) -")
def deep_clean_text(text):
    # 1. Xử lý Header (Nguồn tin & Địa danh) - Phải làm đầu tiên
    # Xóa pattern "CITY (Reuters) -"
    text = re.sub(r'^.*?\(reuters\)\s*[-–—]\s*', '', str(text), flags=re.IGNORECASE)
    # Xóa pattern "CITY -" ở đầu câu
    text = re.sub(r'^.*?\s?[-–—]\s?', '', text, count=1)
    
    # 2. Chuẩn hóa cơ bản
    text = text.lower() # Chuyển về chữ thường
    
    # 3. Xử lý ký tự đặc biệt "rác" trên môi trường web
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Xóa URL
    text = re.sub(r'<.*?>', '', text) # Xóa thẻ HTML
    
    # 4. Xử lý dấu câu và ký tự đặc biệt còn lại
    # Sử dụng string.punctuation để quét sạch các ký tự rác
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 5. Xử lý khoảng trắng và xuống dòng
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['text'] = df['text'].apply(deep_clean_text)

# 5. Tạo 2 phiên bản dữ liệu để thử nghiệm đa kỹ thuật
# Phiên bản 1: Chỉ lấy nội dung (Text Only)
df['text_only'] = df['text']

# Phiên bản 2: Kết hợp Tiêu đề + Nội dung (Title + Text)
df['title_text'] = df['title'] + " " + df['text']

print("Làm sạch hoàn tất. Đã tạo 2 phiên bản: 'text_only' và 'title_text'.")

## 3. Chuẩn hóa ngôn ngữ

In [ ]:
# 1. Khởi tạo Lemmatizer và bộ Stop words chuyên biệt
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Bổ sung các từ chuyên biệt cho tin tức chính trị thường xuất hiện nhiều nhưng ít giá trị phân loại
political_stopwords = {'said', 'also', 'would', 'could', 'told', 'reuters', 'washington', 'statement', 'press'}
stop_words.update(political_stopwords)

def final_preprocess(text):
    # a. Chuyển về chữ thường
    text = str(text).lower()
    
    # b. Loại bỏ dấu câu
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # c. Tách từ, loại bỏ stop words và thực hiện Lemmatization
    words = text.split()
    clean_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(clean_words)

# 2. Áp dụng chuẩn hóa ngôn ngữ cho cột dữ liệu
print("Đang chuẩn hóa ngôn ngữ (Lemmatization)...")
df['text_only_clean'] = df['text_only'].apply(final_preprocess)
df['title_text_clean'] = df['title_text'].apply(final_preprocess)
print("Chuẩn hóa thành công")

## 4. Sản phẩm

In [ ]:
# Lưu vào thư mục processed sau khi đã chuẩn hóa xong
output_path = '../data/processed/processed_data.csv'
os.makedirs('../data/processed', exist_ok=True)

# Lưu đầy đủ các cột để sau này so sánh (text thô và text đã chuẩn hóa)
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"✅ Đã chuẩn hóa xong và lưu tại: {output_path}")
print(f"Cột dữ liệu mới sẵn sàng cho huấn luyện: 'text_only_clean' và 'title_text_clean'")

## 4. Phân tích dữ liệu (EDA)

In [ ]:
# Thiết lập style cho biểu đồ
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)

# 1. Biểu đồ phân bổ nhãn (Countplot)
plt.subplot(1, 2, 1)
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Phân bổ nhãn Tin thật (0) và Tin giả (1)')
plt.xlabel('Nhãn')
plt.ylabel('Số lượng mẫu')

# 2. Biểu đồ Histogram độ dài văn bản (Số lượng từ)
# Tính số lượng từ trong mỗi bài báo
df['text_length'] = df['text'].apply(lambda x: len(x.split()))

plt.subplot(1, 2, 2)
sns.histplot(df['text_length'], bins=50, kde=True, color='blue')
plt.title('Phân bổ độ dài văn bản (Số lượng từ)')
plt.xlabel('Số lượng từ')
plt.ylabel('Tần suất')

plt.tight_layout()
plt.show()

# 3. Biểu đồ so sánh độ dài văn bản giữa Tin thật và Tin giả (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='label', y='text_length', data=df, palette='Set2')
plt.title('So sánh độ dài văn bản theo nhãn')
plt.xlabel('Nhãn (0: True, 1: Fake)')
plt.ylabel('Độ dài văn bản (từ)')
# Giới hạn trục y nếu có outlier quá lớn để biểu đồ dễ nhìn hơn
# plt.ylim(0, 2000) 
plt.show()

### 3. Chia tập dữ liệu và trích xuất đặc trưng

In [ ]:
print("Đang chia tập dữ liệu và trích xuất đặc trưng...")
x = df['clean_text']
y = df['label']

# Chia dữ liệu: 80% để train, 20% để test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# Sử dụng TF-IDF để chuyển đổi văn bản thành ma trận các con số
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2)) # Lấy 5000 từ quan trọng nhất
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

### 4. Huấn luyện với mô hình Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

print("Đang huấn luyện mô hình Logistic Regression ...")
LR = LogisticRegression()
LR.fit(x_train_tfidf, y_train)

pred_LR = LR.predict(x_test_tfidf)
print("\n=== KẾT QUẢ ĐÁNH GIÁ ===")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test, pred_LR) * 100:.2f}%")
print(classification_report(y_test, pred_LR, target_names=["Tin thật (0)", "Tin giả (1)"]))

### 5. Huấn luyện với mô hình Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("Đang huấn luyện mô hình Decision Tree...")
DT = DecisionTreeClassifier()
DT.fit(x_train_tfidf, y_train)

pred_DT = DT.predict(x_test_tfidf)
print("\n=== KẾT QUẢ DECISION TREE ===")
print(f"Độ chính xác: {accuracy_score(y_test, pred_DT) * 100:.2f}%")
print(classification_report(y_test, pred_DT, target_names=["Tin thật (0)", "Tin giả (1)"]))


### 6. Huấn luyện với mô hình Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Đang huấn luyện mô hình Random Forest...")
RF = RandomForestClassifier(random_state=0)
RF.fit(x_train_tfidf, y_train)

pred_RF = RF.predict(x_test_tfidf)
print("\n=== KẾT QUẢ RANDOM FOREST ===")
print(f"Độ chính xác: {accuracy_score(y_test, pred_RF) * 100:.2f}%")
print(classification_report(y_test, pred_RF, target_names=["Tin thật (0)", "Tin giả (1)"]))


### 7. Huấn luyện với mô hình Naive Bayes


In [ ]:
from sklearn.naive_bayes import MultinomialNB

print("Đang huấn luyện mô hình Naive Bayes...")
NB = MultinomialNB()
NB.fit(x_train_tfidf, y_train)

pred_NB = NB.predict(x_test_tfidf)
print("\n=== KẾT QUẢ NAIVE BAYES ===")
print(f"Độ chính xác: {accuracy_score(y_test, pred_NB) * 100:.2f}%")
print(classification_report(y_test, pred_NB, target_names=["Tin thật (0)", "Tin giả (1)"]))


### 8. Huấn luyện với mô hình Support Vector Machine (SVM)


In [ ]:
from sklearn.svm import SVC

print("Đang huấn luyện mô hình SVM (có thể mất vài phút)...")
# Khởi tạo SVM với kernel='linear' và bật probability=True để tính xác suất
SVM = SVC(kernel='linear', probability=True, random_state=0)
SVM.fit(x_train_tfidf, y_train)

pred_SVM = SVM.predict(x_test_tfidf)
print("\n=== KẾT QUẢ SVM ===")
print(f"Độ chính xác: {accuracy_score(y_test, pred_SVM) * 100:.2f}%")
print(classification_report(y_test, pred_SVM, target_names=["Tin thật (0)", "Tin giả (1)"]))


In [ ]:
# --- THỬ NGHIỆM THỰC TẾ VỚI GIAO DIỆN ---
import ipywidgets as widgets
from IPython.display import display

# Tạo giao diện nhập liệu
text_area = widgets.Textarea(
    value='',
    placeholder='Dán nội dung bài báo tiếng Anh vào đây...',
    description='Bài báo:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='150px')
)
button = widgets.Button(description="Dự đoán", button_style='success')
output = widgets.Output()

display(text_area, button, output)

def on_button_clicked(b):
    with output:
        output.clear_output()
        news_text = text_area.value
        if not news_text.strip():
            print("Vui lòng nhập nội dung bài báo.")
            return
        
        # Tiền xử lý và vector hóa
        news_cleaned = preprocess_text(news_text)
        news_tfidf = vectorizer.transform([news_cleaned])
        
        models = {
            'Logistic Regression': LR,
            'Decision Tree': DT,
            'Random Forest': RF,
            'Naive Bayes': NB,
            'SVM': SVM
        }
        
        print(f"{'='*50}")
        print("KẾT QUẢ DỰ ĐOÁN TỪ CÁC MÔ HÌNH:")
        print(f"{'='*50}")
        for name, model in models.items():
            res = model.predict(news_tfidf)[0]
            prob = model.predict_proba(news_tfidf)[0]
            max_prob = max(prob) * 100
            label_str = "Tin giả 🛑" if res == 1 else "Tin thật ✅"
            print(f"- {name:<20}: {label_str:<12} (Xác suất: {max_prob:.2f}%)")
        print(f"{'='*50}")

button.on_click(on_button_clicked)
